<a href="https://colab.research.google.com/github/ArifahZhafirah/PEMBELAJARAN-MESIN/blob/main/JS09/Tugas-LAB01/PEMMESTUGAS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tugas Lab 1

In [1]:
# Install library
!pip install annoy
!pip install faiss-cpu
!pip install hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 7.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=ed97ac0a121f3160617907434fb44182404fb21983ded570a937d08385822c78
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 26.5 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=3e2b1dd04678ced824c499c1d5c60e92ad0cb037347c7e6b8089bb3e3150cebf
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705

In [2]:
import numpy as np
import pandas as pd
import time
import faiss
import hnswlib
from annoy import AnnoyIndex

In [3]:
def run_experiment(n_data, dim, metric):
    X = np.random.random((n_data, dim)).astype(np.float32)
    query = np.random.random((1, dim)).astype(np.float32)
    k = 10

    # Untuk cosine, vektor FAISS dinormalisasi dulu (inner product)
    if metric == 'cosine':
        X_faiss = X.copy()
        query_faiss = query.copy()
        faiss.normalize_L2(X_faiss)
        faiss.normalize_L2(query_faiss)
    else:
        X_faiss = X
        query_faiss = query

    hasil = {}

    # ---------- Annoy ----------
    annoy_metric = 'euclidean' if metric == 'euclidean' else 'angular'
    ann_index = AnnoyIndex(dim, annoy_metric)

    start = time.time()
    for i in range(n_data):
        ann_index.add_item(i, X[i])
    ann_index.build(10)  # 10 trees
    build_time = time.time() - start

    start = time.time()
    ann_index.get_nns_by_vector(query[0], k, include_distances=True)
    query_time = time.time() - start
    hasil['ANNOY'] = (build_time, query_time)

    # ---------- FAISS (Flat Index) ----------
    if metric == 'euclidean':
        faiss_index = faiss.IndexFlatL2(dim)
    else:
        faiss_index = faiss.IndexFlatIP(dim)

    start = time.time()
    faiss_index.add(X_faiss)
    build_time = time.time() - start

    start = time.time()
    faiss_index.search(query_faiss, k)
    query_time = time.time() - start
    hasil['FAISS'] = (build_time, query_time)

    # ---------- HNSW (hnswlib) ----------
    hnsw_space = 'l2' if metric == 'euclidean' else 'cosine'
    hnsw_index = hnswlib.Index(space=hnsw_space, dim=dim)

    start = time.time()
    hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
    hnsw_index.add_items(X)
    build_time = time.time() - start

    hnsw_index.set_ef(50)

    start = time.time()
    hnsw_index.knn_query(query, k=k)
    query_time = time.time() - start
    hasil['HNSW'] = (build_time, query_time)

    return hasil

In [4]:
konfigurasi = [
    (1000, 2, 'euclidean'),
    (1000, 2, 'cosine'),
    (1000, 5, 'euclidean'),
    (1000, 5, 'cosine'),
    (1000000, 2, 'euclidean'),
    (1000000, 2, 'cosine'),
    (1000000, 5, 'euclidean'),
    (1000000, 5, 'cosine'),
]

np.random.seed(42)
rows = []
for n_data, dim, metric in konfigurasi:
    print("Menjalankan:", n_data, dim, metric)
    h = run_experiment(n_data, dim, metric)
    rows.append({
        'Jumlah Data/Dimensi': f"{n_data}/{dim}D ({metric})",
        'ANNOY (build, query)': f"{h['ANNOY'][0]:.4f} s, {h['ANNOY'][1]*1000:.4f} ms",
        'FAISS (build, query)': f"{h['FAISS'][0]:.4f} s, {h['FAISS'][1]*1000:.4f} ms",
        'HNSW (build, query)': f"{h['HNSW'][0]:.4f} s, {h['HNSW'][1]*1000:.4f} ms",
    })

Menjalankan: 1000 2 euclidean
Menjalankan: 1000 2 cosine
Menjalankan: 1000 5 euclidean
Menjalankan: 1000 5 cosine
Menjalankan: 1000000 2 euclidean
Menjalankan: 1000000 2 cosine
Menjalankan: 1000000 5 euclidean
Menjalankan: 1000000 5 cosine


In [5]:
# Tabel hasil percobaan
df_hasil = pd.DataFrame(rows)
df_hasil

,Jumlah Data/Dimensi,"ANNOY (build, query)","FAISS (build, query)","HNSW (build, query)"
0,1000/2D (euclidean),"0.0409 s, 0.1063 ms","0.0001 s, 0.0937 ms","0.0950 s, 0.0763 ms"
1,1000/2D (cosine),"0.0610 s, 0.0858 ms","0.0000 s, 0.0641 ms","0.0818 s, 0.0672 ms"
2,1000/5D (euclidean),"0.0312 s, 0.0904 ms","0.0000 s, 0.0584 ms","0.1167 s, 0.0806 ms"
3,1000/5D (cosine),"0.0424 s, 0.0839 ms","0.0000 s, 0.0820 ms","0.1161 s, 0.0994 ms"
4,1000000/2D (euclidean),"32.2662 s, 0.1335 ms","0.0035 s, 7.5583 ms","132.0545 s, 0.1366 ms"
5,1000000/2D (cosine),"96.6800 s, 0.1581 ms","0.0060 s, 5.3849 ms","175.2083 s, 0.1340 ms"
6,1000000/5D (euclidean),"25.1447 s, 0.1469 ms","0.0242 s, 9.5546 ms","211.0953 s, 0.2139 ms"
7,1000000/5D (cosine),"31.5395 s, 0.1361 ms","0.0185 s, 5.6601 ms","198.6703 s, 0.2325 ms"
